# Phase 3 — Legal Rhetorical Role Prediction with the Phase 1 V4 Hybrid Model

This notebook turns the already-trained Phase 1 hybrid language model into a Phase 3 sentence-sequence classifier. It does **not** train the language model again, download Phase 1 data, or access Hugging Face.

## What it compares

1. **V4 frozen encoder + document BiLSTM:** encode each sentence once with the supplied hybrid checkpoint; train a two-layer BiLSTM over each document's sentence vectors. This follows the best-performing Phase 3 pattern already observed with V1: document context improved the development mean from about 0.295 (sentence classifier) to 0.367 (context BiLSTM).
2. **Historical reference check:** replay the original V1 context head and, when attached, the later accepted V1 safe-retrain head. The safe-retrain checkpoint is the strongest previous development result (about 0.4107 mean). V4 must clear that stronger comparison before it can replace the selected submission. A fixed 50:50 probability blend is evaluated and selected only on development metrics; test prediction files are provenance checks only.

The supplied Phase 1 checkpoint is called “V4” in this project because it is the fourth Phase 1 model. Its internal config says `architecture_version=V2`; the actual architecture is the 12-layer hybrid global/local decoder in the checkpoint. The notebook uses the checkpoint's own config and the original model definition.

## Kaggle inputs

Required files, found recursively under `/kaggle/input`:

- `hybrid_model.pt` (the model-only file from `cs781_phase1_hybrid/final_model/`)
- `bpe_en_25k.json`, `label_vocab.json`
- `best_04311_prediction.json` (saved public incumbent; required for the protected fallback)
- `CL_train.json`, `CL_dev.json`, `CL_test.json`
- `IT_train.json`, `IT_dev.json`, `IT_test.json`

Optional V1 comparison files (attach these to replay the strongest prior head and evaluate a blend):

- `v1_model.pt`
- `best_phase3_v1_context_bilstm.pt`
- `selected_phase3_context_model.pt` (the later V1 safe-retrain context head)
- `phase3_v1_safe_retrain_prediction.json` (rename the safe-retrain `prediction.json` to avoid filename collisions)

The strongest prior development result was the accepted V1 safe-retrain run (mean 0.410681; CL 0.387006, IT 0.434356). Those audited development values remain the guard baseline even when the checkpoint is not attached. The 0.4311 incumbent prediction is required as a protected fallback. To replay the safe model and evaluate a blend, also attach the V1 Phase 1 model, original V1 context head, safe-retrain context head, and safely renamed safe-retrain prediction file. Turn on a Kaggle GPU before running the full notebook.

## Metric and leakage control

Selection uses the unweighted mean of **fixed 13-label Macro-F1 on CL development** and **fixed 13-label Macro-F1 on IT development**. The gold-supported-label score is diagnostic only. The supplied test split is used only after model selection to produce the required predictions. It is never used to select epochs, blend weights, or models.

The dataset is small: 40 training, 5 development, and 5 test documents per domain. The development estimate can therefore move noticeably with a few documents. A score above 0.50 is not promised; the hidden Codabench score is only known after submission.

## Research references

- The LegalEval corpus defines 13 rhetorical roles and frames the task as semantic segmentation of legal judgments: [Malik et al. (2022)](https://aclanthology.org/2022.nllp-1.13/).
- Prior LegalEval systems explicitly model sentence sequence context with BiLSTMs: [Jain et al. (2023)](https://aclanthology.org/2023.semeval-1.103/) and [Chen et al. (2023)](https://aclanthology.org/2023.semeval-1.286/). These papers motivate the context layer; they do not guarantee this project's score.
- The V4 Phase 1 checkpoint uses grouped-query attention; see [Ainslie et al. (2023)](https://aclanthology.org/2023.emnlp-main.298/).
- LoRA and LoRMA are documented methods ([LoRA](https://arxiv.org/abs/2106.09685), [LoRMA](https://aclanthology.org/2025.findings-acl.527/)). They are not repeated in this run: earlier project experiments did not show a reliable gain over the simpler sequence-context approach, and the next test should isolate the Phase 1 V4 backbone.


## 1. Environment, Kaggle input discovery, and output folder

The input lookup checks filenames and rejects ambiguous, different copies rather than silently loading an arbitrary checkpoint or split.

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import shutil
import time
import zipfile
from collections import Counter, defaultdict
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/phase_3_v4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_ROOT.exists():
    raise FileNotFoundError("This notebook expects Kaggle inputs under /kaggle/input.")

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def locate_input(filename, required=True):
    matches = sorted(p for p in INPUT_ROOT.rglob(filename) if p.is_file())
    if not matches:
        if required:
            raise FileNotFoundError(
                f"Required input {filename!r} was not found. Attach the Kaggle dataset containing it."
            )
        return None
    if len(matches) == 1:
        return matches[0]
    sizes = {p.stat().st_size for p in matches}
    if len(sizes) == 1 and next(iter(sizes)) <= 10_000_000:
        hashes = {_sha256(p) for p in matches}
        if len(hashes) == 1:
            print(f"Identical duplicate copies of {filename}; using {matches[0]}")
            return matches[0]
    raise RuntimeError(
        f"Found ambiguous copies of {filename}:\n" + "\n".join(map(str, matches))
    )

REQUIRED_FILES = [
    "hybrid_model.pt", "bpe_en_25k.json", "label_vocab.json",
    "best_04311_prediction.json",
    "CL_train.json", "CL_dev.json", "CL_test.json",
    "IT_train.json", "IT_dev.json", "IT_test.json",
]
FILES = {name: locate_input(name, required=True) for name in REQUIRED_FILES}
FILES["v1_model.pt"] = locate_input("v1_model.pt", required=False)
FILES["best_phase3_v1_context_bilstm.pt"] = locate_input(
    "best_phase3_v1_context_bilstm.pt", required=False
)
FILES["selected_phase3_context_model.pt"] = locate_input(
    "selected_phase3_context_model.pt", required=False
)
FILES["phase3_v1_safe_retrain_prediction.json"] = locate_input(
    "phase3_v1_safe_retrain_prediction.json", required=False
)

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. The notebook can run on CPU, but full feature extraction will be much slower.")
print("Output:", OUTPUT_DIR)
print("Resolved inputs:")
for name, path in FILES.items():
    print(f"  {name}: {path if path is not None else 'not attached (optional)'}")


## 2. Read and validate the supplied Phase 3 splits

Training/development data are document lists with `text` and `labels`; test data are ID-to-sentence-list mappings. Both forms are normalized explicitly. The test labels are not expected or read.

In [ ]:
def read_json(path):
    with open(path, "r", encoding="utf-8") as stream:
        return json.load(stream)

def normalize_documents(obj, split_name):
    docs = []
    if isinstance(obj, list):
        iterator = ((None, item) for item in obj)
    elif isinstance(obj, dict):
        iterator = obj.items()
    else:
        raise TypeError(f"{split_name} must be a document list or an ID-to-document mapping")

    for key, value in iterator:
        if isinstance(value, dict):
            doc_id = value.get("id", key)
            sentences = value.get("text", value.get("sentences"))
            labels = value.get("labels")
        elif isinstance(value, list) and key is not None:
            doc_id, sentences, labels = key, value, None
        else:
            raise ValueError(f"{split_name}: unsupported record for document {key!r}")
        if doc_id is None:
            raise ValueError(f"{split_name}: a document has no ID")
        if not isinstance(sentences, list) or not all(isinstance(s, str) for s in sentences):
            raise TypeError(f"{split_name}/{doc_id}: text must be a list of sentence strings")
        if labels is not None and (not isinstance(labels, list) or not all(isinstance(x, str) for x in labels)):
            raise TypeError(f"{split_name}/{doc_id}: labels must be a list of strings")
        docs.append({"id": str(doc_id), "text": sentences, "labels": labels})
    return docs

LABEL_TO_ID = read_json(FILES["label_vocab.json"])
if len(LABEL_TO_ID) != 13 or sorted(LABEL_TO_ID.values()) != list(range(13)):
    raise ValueError("label_vocab.json must map exactly 13 labels to IDs 0 through 12")
ID_TO_LABEL = {int(index): label for label, index in LABEL_TO_ID.items()}
NUM_LABELS = len(LABEL_TO_ID)

SPLITS = {domain: {} for domain in ("CL", "IT")}
for domain in ("CL", "IT"):
    for split in ("train", "dev", "test"):
        name = f"{domain}_{split}.json"
        SPLITS[domain][split] = normalize_documents(read_json(FILES[name]), name)

EXPECTED_DOCS = {"train": 40, "dev": 5, "test": 5}
for domain in ("CL", "IT"):
    seen_ids = set()
    for split in ("train", "dev", "test"):
        docs = SPLITS[domain][split]
        if len(docs) != EXPECTED_DOCS[split]:
            raise ValueError(f"{domain}_{split}: expected {EXPECTED_DOCS[split]} documents, found {len(docs)}")
        ids = [d["id"] for d in docs]
        if len(set(ids)) != len(ids) or seen_ids.intersection(ids):
            raise ValueError(f"{domain}_{split}: duplicate document IDs or overlap between supplied splits")
        seen_ids.update(ids)
        labeled = split != "test"
        for doc in docs:
            labels = doc["labels"]
            if labeled and labels is None:
                raise ValueError(f"{domain}_{split}/{doc['id']}: labels are missing")
            if not labeled and labels is not None:
                raise ValueError(f"{domain}_{split}/{doc['id']}: test labels must not be present")
            if labels is not None:
                if len(labels) != len(doc["text"]):
                    raise ValueError(f"{domain}_{split}/{doc['id']}: sentence and label counts differ")
                unknown = sorted(set(labels) - set(LABEL_TO_ID))
                if unknown:
                    raise ValueError(f"{domain}_{split}/{doc['id']}: unknown labels {unknown}")

all_test_ids = [doc["id"] for domain in ("CL", "IT") for doc in SPLITS[domain]["test"]]
if len(set(all_test_ids)) != len(all_test_ids):
    raise ValueError("CL and IT test document IDs must be unique for the Codabench mapping format")

print("Validated document and sentence counts:")
for domain in ("CL", "IT"):
    for split in ("train", "dev", "test"):
        docs = SPLITS[domain][split]
        n_sent = sum(len(d["text"]) for d in docs)
        print(f"  {domain} {split:5s}: {len(docs):2d} documents, {n_sent:5d} sentences")

TRAIN_SUPPORT = np.zeros(NUM_LABELS, dtype=np.int64)
for domain in ("CL", "IT"):
    for doc in SPLITS[domain]["train"]:
        TRAIN_SUPPORT += np.bincount([LABEL_TO_ID[y] for y in doc["labels"]], minlength=NUM_LABELS)
print("Combined train label support:")
for label_id in range(NUM_LABELS):
    print(f"  {ID_TO_LABEL[label_id]:28s} {TRAIN_SUPPORT[label_id]:5d}")

def metrics_from_ids(gold, pred):
    gold = np.asarray(gold, dtype=np.int64)
    pred = np.asarray(pred, dtype=np.int64)
    if gold.shape != pred.shape or gold.ndim != 1:
        raise ValueError("Metric inputs must be aligned one-dimensional label arrays")
    cm = np.zeros((NUM_LABELS, NUM_LABELS), dtype=np.int64)
    if len(gold):
        np.add.at(cm, (gold, pred), 1)
    tp = np.diag(cm).astype(np.float64)
    support = cm.sum(axis=1)
    predicted = cm.sum(axis=0)
    precision = np.divide(tp, predicted, out=np.zeros_like(tp), where=predicted > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    f1 = np.divide(2 * precision * recall, precision + recall,
                   out=np.zeros_like(tp), where=(precision + recall) > 0)
    supported = support > 0
    return {
        "macro_f1_fixed_13": float(f1.mean()),
        "macro_f1_gold_supported_diagnostic": float(f1[supported].mean()) if supported.any() else 0.0,
        "gold_supported_label_count": int(supported.sum()),
        "support_by_label": {ID_TO_LABEL[i]: int(support[i]) for i in range(NUM_LABELS)},
        "predicted_by_label": {ID_TO_LABEL[i]: int(predicted[i]) for i in range(NUM_LABELS)},
        "f1_by_label": {ID_TO_LABEL[i]: float(f1[i]) for i in range(NUM_LABELS)},
        "confusion_matrix": cm.tolist(),
    }

def metrics_for_domain(logits_by_id, docs):
    gold, pred = [], []
    for doc in docs:
        if doc["labels"] is None:
            continue
        logits = logits_by_id[doc["id"]]
        pred.extend(logits.argmax(-1).tolist())
        gold.extend(LABEL_TO_ID[y] for y in doc["labels"])
    return metrics_from_ids(gold, pred)

def mean_domain_score(domain_metrics):
    return float(np.mean([domain_metrics[d]["macro_f1_fixed_13"] for d in ("CL", "IT")]))


## 3. Read the Phase 1 V4 checkpoint config

The model-only checkpoint is used; `best.pt` and `latest.pt` are not needed because they contain Phase 1 optimizer/resume state. Nothing below starts Phase 1 training.

In [ ]:
def torch_load_safe(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:  # compatibility with older Kaggle PyTorch builds
        return torch.load(path, map_location="cpu")

V4_CHECKPOINT = torch_load_safe(FILES["hybrid_model.pt"])
if not isinstance(V4_CHECKPOINT, dict) or "model_state_dict" not in V4_CHECKPOINT:
    raise ValueError("hybrid_model.pt must contain model_state_dict and config")
V4_CONFIG = V4_CHECKPOINT.get("config", {})
EXPECTED_CONFIG = {
    "vocab_size": 25000, "n_embd": 768, "n_layer": 12,
    "n_head": 12, "n_kv_head": 4, "ffn_multiplier": 4,
    "context_length": 1024, "local_attention_window": 256,
}
for key, expected in EXPECTED_CONFIG.items():
    if key not in V4_CONFIG or int(V4_CONFIG[key]) != expected:
        raise ValueError(f"V4 checkpoint config mismatch for {key}: {V4_CONFIG.get(key)} vs {expected}")
if list(map(int, V4_CONFIG.get("global_attention_layers", []))) != [0, 4, 8, 11]:
    raise ValueError("Unexpected V4 global-attention layer pattern")
if int(V4_CONFIG["vocab_size"]) != NUM_LABELS and int(V4_CONFIG["vocab_size"]) != 25000:
    raise ValueError("Checkpoint vocabulary size does not match the supplied BPE-25k tokenizer")

GLOBAL_ATTENTION_LAYERS = list(map(int, V4_CONFIG["global_attention_layers"]))
LOCAL_ATTN_WINDOW = int(V4_CONFIG["local_attention_window"])
print("Checkpoint config:", json.dumps(V4_CONFIG, indent=2))
print("Phase 1 validation loss in model-only checkpoint:", V4_CHECKPOINT.get("best_val_loss"))
print("Phase 1 validation perplexity in model-only checkpoint:", V4_CHECKPOINT.get("best_val_perplexity"))


## 4. Original V4 hybrid model implementation

The following is copied from the model-definition cell in the supplied `v4.ipynb`. Its parameter names and shapes are kept intact so the Phase 1 state dict can be loaded strictly.

In [ ]:
# 9. V2 Lightweight Hybrid Decoder-Only Transformer
#
# V2 preserves the successful V1 components:
# - RMSNorm
# - RoPE
# - Grouped Query Attention (GQA)
# - Causal self-attention
# - Standard GELU FFN
# - Residual connections
# - KV cache
#
# New architectural component:
# - Hybrid Global + Local Causal Attention
#
# Global layers:
#   Full causal attention over the complete sequence
#
# Local layers:
#   Causal attention restricted to a fixed local window
#
# The hybrid design is motivated by the course discussion of
# sparse/local attention and hybrid models that interleave
# full and sparse attention layers.

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------
# RMSNorm
# ---------------------------------------------------------

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(
            dim=-1,
            keepdim=True
        )

        x = x * torch.rsqrt(
            rms + self.eps
        )

        return self.weight * x


# ---------------------------------------------------------
# Rotary Positional Embeddings
# ---------------------------------------------------------

class RotaryEmbedding(nn.Module):
    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=10000
    ):
        super().__init__()

        self.head_dim = head_dim
        self.max_seq_len = max_seq_len

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2
                ).float()
                / head_dim
            )
        )

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False
        )

        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        positions = torch.arange(
            seq_len,
            device=self.inv_freq.device
        ).float()

        freqs = torch.outer(
            positions,
            self.inv_freq
        )

        self.register_buffer(
            "cos_cached",
            freqs.cos(),
            persistent=False
        )

        self.register_buffer(
            "sin_cached",
            freqs.sin(),
            persistent=False
        )

    def forward(
        self,
        q,
        k,
        position_offset=0
    ):
        seq_len = q.size(-2)

        required_len = (
            position_offset + seq_len
        )

        if required_len > self.max_seq_len:
            self._build_cache(required_len)
            self.max_seq_len = required_len

        cos = self.cos_cached[
            position_offset:
            position_offset + seq_len
        ]

        sin = self.sin_cached[
            position_offset:
            position_offset + seq_len
        ]

        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)

        q = self._apply_rotary(
            q,
            cos,
            sin
        )

        k = self._apply_rotary(
            k,
            cos,
            sin
        )

        return q, k

    @staticmethod
    def _apply_rotary(x, cos, sin):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]

        rotated = torch.stack(
            (-x2, x1),
            dim=-1
        ).flatten(-2)

        cos_full = torch.repeat_interleave(
            cos,
            2,
            dim=-1
        )

        sin_full = torch.repeat_interleave(
            sin,
            2,
            dim=-1
        )

        return (
            x * cos_full
            + rotated * sin_full
        )


# ---------------------------------------------------------
# Hybrid Grouped Query Attention
# ---------------------------------------------------------

class GQAAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        attention_type="global",
        local_window=256,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0

        assert attention_type in {
            "global",
            "local"
        }

        assert local_window > 0

        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head

        self.kv_repeat = (
            n_head // n_kv_head
        )

        self.attention_type = attention_type
        self.local_window = local_window

        self.q_proj = nn.Linear(
            n_embd,
            n_head * self.head_dim,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.rope = RotaryEmbedding(
            head_dim=self.head_dim,
            max_seq_len=max_seq_len
        )

        self.dropout = dropout

    def _build_attention_mask(
        self,
        query_length,
        key_length,
        query_position_offset,
        device
    ):
        """
        Build a boolean attention mask.

        True  = attention is allowed
        False = attention is blocked

        For global attention:
            causal mask

        For local attention:
            causal + fixed sliding window
        """

        query_positions = (
            query_position_offset
            + torch.arange(
                query_length,
                device=device
            )
        )

        key_positions = torch.arange(
            key_length,
            device=device
        )

        # Causal condition:
        # query at position q can only see keys <= q.
        causal_mask = (
            key_positions.unsqueeze(0)
            <= query_positions.unsqueeze(1)
        )

        if self.attention_type == "global":
            return causal_mask

        # Local causal attention:
        # q can see at most the previous
        # local_window - 1 tokens plus itself.
        local_mask = (
            key_positions.unsqueeze(0)
            >= (
                query_positions.unsqueeze(1)
                - self.local_window
                + 1
            )
        )

        return causal_mask & local_mask

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        B, T, C = x.shape

        # -------------------------------------------------
        # Q, K, V projections
        # -------------------------------------------------

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.n_head,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        # -------------------------------------------------
        # KV cache information
        # -------------------------------------------------

        past_len = 0

        if past_key_value is not None:
            past_k, past_v = past_key_value
            past_len = past_k.size(-2)

        # -------------------------------------------------
        # RoPE
        # -------------------------------------------------

        q, k = self.rope(
            q,
            k,
            position_offset=past_len
        )

        # -------------------------------------------------
        # Append new K/V to cache
        # -------------------------------------------------

        if past_key_value is not None:
            k = torch.cat(
                [past_k, k],
                dim=-2
            )

            v = torch.cat(
                [past_v, v],
                dim=-2
            )

        present = None

        if use_cache:
            present = (k, v)

        # -------------------------------------------------
        # GQA
        # -------------------------------------------------

        if self.kv_repeat > 1:
            k = k.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

            v = v.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

        # -------------------------------------------------
        # Hybrid attention mask
        # -------------------------------------------------

        key_length = k.size(-2)

        attn_mask = self._build_attention_mask(
            query_length=T,
            key_length=key_length,
            query_position_offset=past_len,
            device=x.device
        )

        # -------------------------------------------------
        # Scaled dot-product attention
        # -------------------------------------------------

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        # -------------------------------------------------
        # Output projection
        # -------------------------------------------------

        y = y.transpose(
            1,
            2
        ).contiguous()

        y = y.view(
            B,
            T,
            C
        )

        y = self.out_proj(y)

        return y, present


# ---------------------------------------------------------
# Standard GELU FFN
# ---------------------------------------------------------

class FeedForward(nn.Module):
    def __init__(
        self,
        n_embd,
        multiplier=4,
        dropout=0.0
    ):
        super().__init__()

        hidden_dim = int(
            n_embd * multiplier
        )

        self.fc1 = nn.Linear(
            n_embd,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_embd
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)

        return x


# ---------------------------------------------------------
# Hybrid Transformer Block
# ---------------------------------------------------------

class TransformerBlock(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout,
        layer_index,
        global_attention_layers,
        local_window
    ):
        super().__init__()

        self.layer_index = layer_index

        if layer_index in global_attention_layers:
            attention_type = "global"
        else:
            attention_type = "local"

        self.attention_type = attention_type

        self.norm1 = RMSNorm(n_embd)

        self.attention = GQAAttention(
            n_embd=n_embd,
            n_head=n_head,
            n_kv_head=n_kv_head,
            max_seq_len=max_seq_len,
            attention_type=attention_type,
            local_window=local_window,
            dropout=dropout
        )

        self.norm2 = RMSNorm(n_embd)

        self.ffn = FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        # Pre-norm attention + residual.
        attn_output, present = self.attention(
            self.norm1(x),
            past_key_value=past_key_value,
            use_cache=use_cache
        )

        x = x + attn_output

        # Pre-norm FFN + residual.
        x = x + self.ffn(
            self.norm2(x)
        )

        return x, present


# ---------------------------------------------------------
# V2 Lightweight Hybrid Legal Language Model
# ---------------------------------------------------------

class HybridLegalLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embd,
        n_layer,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.n_embd = n_embd

        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                n_embd=n_embd,
                n_head=n_head,
                n_kv_head=n_kv_head,
                max_seq_len=max_seq_len,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout,
                layer_index=layer_index,
                global_attention_layers=GLOBAL_ATTENTION_LAYERS,
                local_window=LOCAL_ATTN_WINDOW
            )
            for layer_index in range(n_layer)
        ])

        self.final_norm = RMSNorm(
            n_embd
        )

        # Weight tying retained from V1.
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False
    ):
        x = self.token_embedding(
            input_ids
        )

        presents = [] if use_cache else None

        for layer_index, block in enumerate(
            self.blocks
        ):
            past = None

            if past_key_values is not None:
                past = past_key_values[
                    layer_index
                ]

            x, present = block(
                x,
                past_key_value=past,
                use_cache=use_cache
            )

            if use_cache:
                presents.append(
                    present
                )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits, presents

## 5. Strict checkpoint load and hidden-state equivalence check

The hidden extraction below follows the original forward path through the same embeddings, transformer blocks, and final RMSNorm; it omits only the 25,000-way language-model projection.

In [ ]:
def new_hybrid_model(config):
    return HybridLegalLM(
        vocab_size=int(config["vocab_size"]),
        n_embd=int(config["n_embd"]),
        n_layer=int(config["n_layer"]),
        n_head=int(config["n_head"]),
        n_kv_head=int(config["n_kv_head"]),
        max_seq_len=int(config["context_length"]),
        ffn_multiplier=int(config["ffn_multiplier"]),
        dropout=float(config.get("dropout", 0.0)),
    )

def extract_final_hidden(backbone, input_ids):
    x = backbone.token_embedding(input_ids)
    for block in backbone.blocks:
        x, _ = block(x, past_key_value=None, use_cache=False)
    return backbone.final_norm(x)

V4_BACKBONE = new_hybrid_model(V4_CONFIG)
v4_load_result = V4_BACKBONE.load_state_dict(V4_CHECKPOINT["model_state_dict"], strict=True)
assert not v4_load_result.missing_keys and not v4_load_result.unexpected_keys
V4_BACKBONE = V4_BACKBONE.to(DEVICE).eval()
for parameter in V4_BACKBONE.parameters():
    parameter.requires_grad_(False)

UNIQUE_PARAMETERS = sum(parameter.numel() for parameter in V4_BACKBONE.parameters())
assert UNIQUE_PARAMETERS == 94_762_752, f"Unexpected parameter count: {UNIQUE_PARAMETERS:,}"
probe = torch.randint(0, 25000, (2, 20), device=DEVICE)
with torch.inference_mode():
    logits, _ = V4_BACKBONE(probe, use_cache=False)
    hidden = extract_final_hidden(V4_BACKBONE, probe)
    projected_logits = V4_BACKBONE.lm_head(hidden)
max_difference = float((logits - projected_logits).abs().max())
assert torch.allclose(logits, projected_logits, atol=1e-5, rtol=1e-5)
assert torch.isfinite(logits).all()
print("Strict V4 checkpoint load: passed")
print("Unique V4 parameters:", f"{UNIQUE_PARAMETERS:,}")
print("Hidden path equals original forward path; maximum logit difference:", max_difference)


## 6. Reuse the exact BPE-25k tokenizer and tokenize sentences

The Phase 3 pipeline uses the same vocabulary and byte-level merge rules as Phase 1 and the successful V1 context baseline. Sentences are capped at 512 tokens, as in that V1 pipeline; no BOS or EOS token is added, and the last real token represents the sentence.

In [ ]:
BPE_DATA = read_json(FILES["bpe_en_25k.json"])
BPE_ITOS = BPE_DATA["itos"]
BPE_MERGES = BPE_DATA["merges"]
if int(BPE_DATA.get("vocab_size", len(BPE_ITOS))) != 25000 or len(BPE_ITOS) != 25000:
    raise ValueError("The supplied tokenizer is not the expected 25k vocabulary")
BPE_STOI = {token: index for index, token in enumerate(BPE_ITOS)}
if len(BPE_STOI) != 25000:
    raise ValueError("The BPE vocabulary contains duplicate token strings")
MERGE_TO_ID = {
    (BPE_STOI[left], BPE_STOI[right]): 261 + merge_index
    for merge_index, (left, right) in enumerate(BPE_MERGES)
}
PAD_ID = BPE_STOI["<pad>"]
UNK_ID = BPE_STOI["<unk>"]

class ByteLevelBPE25k:
    def __init__(self, itos, stoi, merge_to_id):
        self.itos, self.stoi, self.merge_to_id = itos, stoi, merge_to_id
        self.vocab_size = len(itos)

    def _encode_word(self, word):
        sequence = [4 + byte for byte in word.encode("utf-8")] + [self.stoi["<wb>"]]
        while len(sequence) > 1:
            best_id, best_position = None, None
            for position in range(len(sequence) - 1):
                token_id = self.merge_to_id.get((sequence[position], sequence[position + 1]))
                if token_id is not None and (best_id is None or token_id < best_id):
                    best_id, best_position = token_id, position
            if best_position is None:
                break
            sequence = sequence[:best_position] + [best_id] + sequence[best_position + 2:]
        return sequence

    def encode(self, text):
        result = []
        for word in str(text).split():
            result.extend(self._encode_word(word))
        return result

TOKENIZER = ByteLevelBPE25k(BPE_ITOS, BPE_STOI, MERGE_TO_ID)
MAX_SENTENCE_TOKENS = 512
FEATURE_BATCH_SIZE = 4 if DEVICE.type == "cuda" else 1
TRUNCATION_COUNTS = {}

def prepare_tokenized_documents(docs, split_key):
    prepared, truncated = [], 0
    for doc in docs:
        token_rows = []
        for sentence in doc["text"]:
            ids = TOKENIZER.encode(sentence)
            if not ids:
                ids = [UNK_ID]
            if len(ids) > MAX_SENTENCE_TOKENS:
                truncated += 1
                ids = ids[:MAX_SENTENCE_TOKENS]
            token_rows.append(ids)
        labels = None if doc["labels"] is None else torch.tensor(
            [LABEL_TO_ID[label] for label in doc["labels"]], dtype=torch.long
        )
        prepared.append({"id": doc["id"], "input_ids": token_rows, "labels": labels})
    TRUNCATION_COUNTS[split_key] = truncated
    return prepared

TOKENIZED = {domain: {} for domain in ("CL", "IT")}
for domain in ("CL", "IT"):
    for split in ("train", "dev", "test"):
        key = f"{domain}_{split}"
        TOKENIZED[domain][split] = prepare_tokenized_documents(SPLITS[domain][split], key)
        print(key, "sentences", sum(len(d["input_ids"]) for d in TOKENIZED[domain][split]),
              "truncated at 512 tokens", TRUNCATION_COUNTS[key])
assert TOKENIZER.vocab_size == int(V4_CONFIG["vocab_size"])
assert len(TOKENIZER.encode("The Supreme Court considered the appeal.")) > 0


## 7. Extract frozen V4 sentence vectors

For each sentence, we take the final hidden state at its last real token. Because the decoder is causal, that position has processed the preceding sentence tokens. A document BiLSTM then reads these sentence vectors in order. Padding is added only on the right; a sentence's real positions cannot attend to later padding under the causal mask.

In [ ]:
def autocast_context():
    if DEVICE.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()

def extract_document_features(backbone, prepared_docs, domain, split):
    rows = []
    flat = []
    for doc_index, doc in enumerate(prepared_docs):
        for sentence_index, ids in enumerate(doc["input_ids"]):
            flat.append((doc_index, sentence_index, ids))
        rows.append({"id": doc["id"], "domain": domain, "labels": doc["labels"], "features": None})
    vectors_by_doc = [[] for _ in prepared_docs]
    for start in range(0, len(flat), FEATURE_BATCH_SIZE):
        batch = flat[start:start + FEATURE_BATCH_SIZE]
        lengths = torch.tensor([len(item[2]) for item in batch], dtype=torch.long)
        max_len = int(lengths.max())
        input_ids = torch.full((len(batch), max_len), PAD_ID, dtype=torch.long)
        for row_index, (_, _, ids) in enumerate(batch):
            input_ids[row_index, :len(ids)] = torch.tensor(ids, dtype=torch.long)
        input_ids = input_ids.to(DEVICE, non_blocking=True)
        with torch.inference_mode(), autocast_context():
            hidden = extract_final_hidden(backbone, input_ids)
        positions = lengths.to(DEVICE) - 1
        batch_rows = torch.arange(len(batch), device=DEVICE)
        vectors = hidden[batch_rows, positions].float().cpu()
        for vector, (doc_index, sentence_index, _) in zip(vectors, batch):
            vectors_by_doc[doc_index].append((sentence_index, vector))
        if (start // FEATURE_BATCH_SIZE + 1) % 250 == 0:
            print(f"{domain} {split}: encoded {min(start + len(batch), len(flat)):,}/{len(flat):,} sentences")
    for doc_index, row in enumerate(rows):
        ordered = sorted(vectors_by_doc[doc_index], key=lambda item: item[0])
        row["features"] = torch.stack([vector for _, vector in ordered])
    print(f"{domain} {split}: {len(rows)} documents, {len(flat)} sentence vectors")
    return rows

V4_FEATURES = {domain: {} for domain in ("CL", "IT")}
feature_start = time.time()
for domain in ("CL", "IT"):
    for split in ("train", "dev", "test"):
        V4_FEATURES[domain][split] = extract_document_features(
            V4_BACKBONE, TOKENIZED[domain][split], domain, split
        )
    torch.save(V4_FEATURES[domain], OUTPUT_DIR / f"v4_{domain.lower()}_sentence_features.pt")
print("V4 frozen feature extraction minutes:", round((time.time() - feature_start) / 60, 2))
del V4_BACKBONE, V4_CHECKPOINT
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


## 8. Train the V4 document-context classifier

This reuses the successful V1 head design: a two-layer, bidirectional LSTM with 256 hidden units per direction. Training sees labeled training documents only. Train-only class weights use the same square-root inverse-frequency rule and cap as the V1 context experiment. Each update pairs one CL document with one IT document, and the loss averages the two domain losses.

In [ ]:
class ContextBiLSTM(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, n_layers=2, n_labels=13, dropout=0.25):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=n_layers, batch_first=True,
                            bidirectional=True, dropout=dropout if n_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, n_labels)

    def forward(self, features, lengths):
        packed = pack_padded_sequence(features, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(packed_output, batch_first=True,
                                        total_length=features.shape[1])
        return self.classifier(self.dropout(output))

def collate_documents(documents):
    features = pad_sequence([row["features"] for row in documents], batch_first=True, padding_value=0.0)
    lengths = torch.tensor([len(row["features"]) for row in documents], dtype=torch.long)
    labels = None
    if documents[0]["labels"] is not None:
        labels = pad_sequence([row["labels"] for row in documents], batch_first=True, padding_value=-100)
    return features, lengths, labels, [row["id"] for row in documents], [row["domain"] for row in documents]

def balanced_train_class_weights():
    cl_labels = [LABEL_TO_ID[y] for d in SPLITS["CL"]["train"] for y in d["labels"]]
    it_labels = [LABEL_TO_ID[y] for d in SPLITS["IT"]["train"] for y in d["labels"]]
    cl_counts = np.bincount(cl_labels, minlength=NUM_LABELS).astype(np.float64)
    it_counts = np.bincount(it_labels, minlength=NUM_LABELS).astype(np.float64)
    n_domain = min(len(cl_labels), len(it_labels))
    expected = (cl_counts / len(cl_labels) + it_counts / len(it_labels)) * n_domain
    weights = np.zeros(NUM_LABELS, dtype=np.float32)
    present = expected > 0
    weights[present] = (expected[present].mean() / expected[present]) ** 0.5
    weights[present] = np.minimum(weights[present], 3.0)
    return expected, weights

EXPECTED_BALANCED_COUNTS, CLASS_WEIGHTS = balanced_train_class_weights()
CLASS_WEIGHTS_T = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32, device=DEVICE)
print("Train-only class weights:")
print({ID_TO_LABEL[i]: round(float(CLASS_WEIGHTS[i]), 3) for i in range(NUM_LABELS)})

def make_paired_train_batches(epoch):
    cl_docs, it_docs = V4_FEATURES["CL"]["train"], V4_FEATURES["IT"]["train"]
    n = min(len(cl_docs), len(it_docs))
    rng = np.random.default_rng(SEED + 1000 + epoch)
    cl_idx = rng.choice(len(cl_docs), size=n, replace=False)
    it_idx = rng.choice(len(it_docs), size=n, replace=False)
    order = rng.permutation(n)
    for index in order:
        yield collate_documents([cl_docs[int(cl_idx[index])], it_docs[int(it_idx[index])]])

@torch.inference_mode()
def predict_document_logits(model, documents):
    model.eval()
    predictions = {}
    for doc in documents:
        features = doc["features"].unsqueeze(0).to(DEVICE)
        lengths = torch.tensor([len(doc["features"])], dtype=torch.long, device=DEVICE)
        predictions[doc["id"]] = model(features, lengths)[0].float().cpu()
    return predictions

@torch.inference_mode()
def evaluate_context_model(model, domain, documents):
    logits = predict_document_logits(model, documents)
    return metrics_for_domain(logits, SPLITS[domain]["dev"]), logits

V4_CONTEXT = ContextBiLSTM(n_labels=NUM_LABELS).to(DEVICE)
V4_OPTIMIZER = torch.optim.AdamW(V4_CONTEXT.parameters(), lr=1e-3, weight_decay=1e-3)
MAX_CONTEXT_EPOCHS = 12
CONTEXT_PATIENCE = 3
BEST_CONTEXT_PATH = OUTPUT_DIR / "best_phase3_v4_context_bilstm.pt"
history = []
best_mean = -1.0
best_epoch = None
stale_epochs = 0
best_state = None
ctx_start = time.time()

for epoch in range(1, MAX_CONTEXT_EPOCHS + 1):
    V4_CONTEXT.train()
    total_loss, steps = 0.0, 0
    for features, lengths, labels, _, domains in make_paired_train_batches(epoch):
        features = features.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        lengths = lengths.to(DEVICE, non_blocking=True)
        V4_OPTIMIZER.zero_grad(set_to_none=True)
        logits = V4_CONTEXT(features, lengths)
        per_label_loss = F.cross_entropy(
            logits.reshape(-1, NUM_LABELS), labels.reshape(-1),
            weight=CLASS_WEIGHTS_T, ignore_index=-100, reduction="none"
        ).reshape_as(labels)
        domain_losses = []
        for domain in ("CL", "IT"):
            row_mask = torch.tensor([d == domain for d in domains], device=DEVICE).unsqueeze(1)
            mask = row_mask & labels.ne(-100)
            if mask.any():
                denominator = CLASS_WEIGHTS_T[labels[mask]].sum().clamp_min(1e-12)
                domain_losses.append(per_label_loss[mask].sum() / denominator)
        if len(domain_losses) != 2:
            raise RuntimeError("Every training batch must contain one document from each domain")
        loss = torch.stack(domain_losses).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(V4_CONTEXT.parameters(), 1.0)
        V4_OPTIMIZER.step()
        total_loss += float(loss.detach())
        steps += 1

    dev_metrics, _ = {}, {}
    for domain in ("CL", "IT"):
        dev_metrics[domain], _ = evaluate_context_model(V4_CONTEXT, domain, V4_FEATURES[domain]["dev"])
    score = mean_domain_score(dev_metrics)
    row = {"epoch": epoch, "mean_batch_loss": total_loss / max(steps, 1),
           "dev": dev_metrics, "mean_domain_macro_f1_fixed_13": score}
    history.append(row)
    print(f"Epoch {epoch:02d}/{MAX_CONTEXT_EPOCHS} | loss {row['mean_batch_loss']:.4f} | "
          f"CL {dev_metrics['CL']['macro_f1_fixed_13']:.4f} | "
          f"IT {dev_metrics['IT']['macro_f1_fixed_13']:.4f} | mean {score:.4f}")
    if score > best_mean:
        best_mean, best_epoch, stale_epochs = score, epoch, 0
        best_state = {key: value.detach().cpu().clone() for key, value in V4_CONTEXT.state_dict().items()}
        torch.save({"context_state_dict": best_state, "label_to_id": LABEL_TO_ID,
                    "best_epoch": best_epoch, "dev_metrics": dev_metrics,
                    "selection_score_mean_domain_macro_f1_fixed_13": best_mean,
                    "class_weights_from_train_only": CLASS_WEIGHTS.tolist(),
                    "encoder": "frozen Phase 1 V4 hybrid_model.pt",
                    "context_head": "2-layer bidirectional LSTM, hidden_dim=256, dropout=0.25"},
                   BEST_CONTEXT_PATH)
    else:
        stale_epochs += 1
        if stale_epochs >= CONTEXT_PATIENCE:
            print("Early stop: development mean did not improve for three epochs.")
            break

V4_CONTEXT.load_state_dict(best_state, strict=True)
V4_CONTEXT.eval()
with open(OUTPUT_DIR / "phase3_v4_training_history.json", "w", encoding="utf-8") as stream:
    json.dump(history, stream, ensure_ascii=False, indent=2)
print("Selected V4 context epoch:", best_epoch)
print("Selected V4 development mean fixed-13 Macro-F1:", round(best_mean, 6))
print("Context training minutes:", round((time.time() - ctx_start) / 60, 2))

V4_DEV_LOGITS, V4_TEST_LOGITS = {}, {}
for domain in ("CL", "IT"):
    V4_DEV_LOGITS.update(predict_document_logits(V4_CONTEXT, V4_FEATURES[domain]["dev"]))
    V4_TEST_LOGITS.update(predict_document_logits(V4_CONTEXT, V4_FEATURES[domain]["test"]))
V4_DEV_METRICS = {
    domain: metrics_for_domain(V4_DEV_LOGITS, SPLITS[domain]["dev"])
    for domain in ("CL", "IT")
}
V4_DEV_MEAN = mean_domain_score(V4_DEV_METRICS)
print("V4 candidate dev metrics:", json.dumps({d: V4_DEV_METRICS[d]["macro_f1_fixed_13"] for d in ("CL", "IT")}, indent=2))
print("V4 candidate dev mean:", round(V4_DEV_MEAN, 6))


## 9. Replay the earlier V1 heads, if their inputs are attached

The original V1 model is loaded with the V1 full-attention layout, then its frozen sentence features are reused to replay both the original context head and the later accepted safe-retrain head. State keys are mapped explicitly and loaded strictly. The safe-retrain head must reproduce its stored development mean within 0.005 to count as the strongest reference. Any blend is selected only by its fixed-weight development score. Test-prediction comparisons are recorded as provenance diagnostics; they never affect selection.

In [ ]:
V1_REFERENCE_AVAILABLE = False
V1_REFERENCE_VERIFIED = False
V1_TEST_MATCHES_SAVED = False
V1_DEV_LOGITS, V1_TEST_LOGITS = {}, {}
V1_DEV_METRICS = None
SAVED_INCUMBENT = None
V1_HEAD_CHECKPOINT = None
SAFE_REFERENCE_AVAILABLE = False
SAFE_REFERENCE_VERIFIED = False
SAFE_TEST_MATCHES_SAVED = False
SAFE_DEV_LOGITS, SAFE_TEST_LOGITS = {}, {}
SAFE_DEV_METRICS = None
SAVED_SAFE_PREDICTION = None

def prediction_json_is_aligned(prediction, test_splits):
    expected = {doc["id"]: len(doc["text"]) for domain in ("CL", "IT")
                for doc in test_splits[domain]["test"]}
    if not isinstance(prediction, dict) or set(prediction) != set(expected):
        return False
    for doc_id, n_sentences in expected.items():
        labels = prediction[doc_id]
        if not isinstance(labels, list) or len(labels) != n_sentences:
            return False
        if any(label not in LABEL_TO_ID for label in labels):
            return False
    return True

if FILES["best_04311_prediction.json"] is not None:
    loaded_incumbent = read_json(FILES["best_04311_prediction.json"])
    if prediction_json_is_aligned(loaded_incumbent, SPLITS):
        SAVED_INCUMBENT = loaded_incumbent
        print("Saved 0.4311 incumbent prediction file passes the test ID/length/label contract.")
    else:
        print("Saved incumbent file is attached but does not match the current test contract; it will not be used.")
if SAVED_INCUMBENT is None:
    raise ValueError("best_04311_prediction.json is required as the protected public incumbent and must match the test IDs, lengths, and labels.")

if FILES["v1_model.pt"] is not None and FILES["best_phase3_v1_context_bilstm.pt"] is not None:
    V1_CHECKPOINT = torch_load_safe(FILES["v1_model.pt"])
    V1_HEAD_CHECKPOINT = torch_load_safe(FILES["best_phase3_v1_context_bilstm.pt"])
    V1_CONFIG = V1_CHECKPOINT.get("config", {})
    V1_EXPECTED = {key: int(V4_CONFIG[key]) for key in
                   ("vocab_size", "n_embd", "n_layer", "n_head", "n_kv_head", "ffn_multiplier", "context_length")}
    for key, expected in V1_EXPECTED.items():
        if int(V1_CONFIG.get(key, -1)) != expected:
            raise ValueError(f"V1 reference checkpoint config mismatch at {key}")
    if V1_HEAD_CHECKPOINT.get("label_to_id") != LABEL_TO_ID:
        raise ValueError("V1 context-head label mapping differs from label_vocab.json")

    # V1 used full causal attention in all 12 layers; set this before construction.
    GLOBAL_ATTENTION_LAYERS = list(range(int(V1_CONFIG["n_layer"])))
    LOCAL_ATTN_WINDOW = int(V1_CONFIG.get("context_length", 1024))
    V1_BACKBONE = new_hybrid_model(V1_CONFIG)
    v1_load_result = V1_BACKBONE.load_state_dict(V1_CHECKPOINT["model_state_dict"], strict=True)
    assert not v1_load_result.missing_keys and not v1_load_result.unexpected_keys
    V1_BACKBONE = V1_BACKBONE.to(DEVICE).eval()
    for parameter in V1_BACKBONE.parameters():
        parameter.requires_grad_(False)
    V1_CONTEXT_FEATURES = {domain: {} for domain in ("CL", "IT")}
    for domain in ("CL", "IT"):
        for split in ("dev", "test"):
            V1_CONTEXT_FEATURES[domain][split] = extract_document_features(
                V1_BACKBONE, TOKENIZED[domain][split], domain, f"V1 reference {split}"
            )
    del V1_BACKBONE, V1_CHECKPOINT
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    V1_CONTEXT = ContextBiLSTM(n_labels=NUM_LABELS).to(DEVICE)
    v1_head_load_result = V1_CONTEXT.load_state_dict(V1_HEAD_CHECKPOINT["context_state_dict"], strict=True)
    assert not v1_head_load_result.missing_keys and not v1_head_load_result.unexpected_keys
    for domain in ("CL", "IT"):
        V1_DEV_LOGITS.update(predict_document_logits(V1_CONTEXT, V1_CONTEXT_FEATURES[domain]["dev"]))
        V1_TEST_LOGITS.update(predict_document_logits(V1_CONTEXT, V1_CONTEXT_FEATURES[domain]["test"]))
    V1_DEV_METRICS = {
        domain: metrics_for_domain(V1_DEV_LOGITS, SPLITS[domain]["dev"])
        for domain in ("CL", "IT")
    }
    V1_DEV_MEAN = mean_domain_score(V1_DEV_METRICS)
    recorded_metrics = V1_HEAD_CHECKPOINT.get("dev_metrics", {})
    replay_differences = {}
    for domain in ("CL", "IT"):
        recorded = recorded_metrics.get(domain, {}).get("macro_f1")
        current = V1_DEV_METRICS[domain]["macro_f1_fixed_13"]
        replay_differences[domain] = None if recorded is None else abs(current - float(recorded))
    V1_REFERENCE_VERIFIED = all(
        replay_differences[d] is not None and replay_differences[d] <= 0.005 for d in ("CL", "IT")
    )

    if SAVED_INCUMBENT is not None:
        reproduced = {}
        for domain in ("CL", "IT"):
            for doc in SPLITS[domain]["test"]:
                reproduced[doc["id"]] = [ID_TO_LABEL[int(i)] for i in V1_TEST_LOGITS[doc["id"]].argmax(-1)]
        n_mismatch = sum(
            reproduced[doc_id] != SAVED_INCUMBENT[doc_id] for doc_id in reproduced
        )
        V1_TEST_MATCHES_SAVED = n_mismatch == 0
        print("V1 recomputation vs saved incumbent documents with a label difference:", n_mismatch)
    print("V1 dev replay absolute differences:", replay_differences)
    print("V1 mean development fixed-13 Macro-F1:", round(V1_DEV_MEAN, 6))
    print("V1 incumbent reproduction verified:", V1_REFERENCE_VERIFIED)
    V1_REFERENCE_AVAILABLE = True

    if FILES["selected_phase3_context_model.pt"] is not None:
        SAFE_CHECKPOINT = torch_load_safe(FILES["selected_phase3_context_model.pt"])
        if SAFE_CHECKPOINT.get("label_to_id") != LABEL_TO_ID:
            raise ValueError("V1 safe-retrain label mapping differs from label_vocab.json")
        if not SAFE_CHECKPOINT.get("candidate_accepted", False):
            raise ValueError("Attached V1 safe-retrain checkpoint is not marked as accepted")
        safe_config = SAFE_CHECKPOINT.get("v1_config", {})
        for key in ("vocab_size", "n_embd", "n_layer", "n_head", "n_kv_head", "ffn_multiplier", "context_length"):
            if key in safe_config and int(safe_config[key]) != int(V1_CONFIG[key]):
                raise ValueError(f"V1 safe-retrain checkpoint config mismatch at {key}")
        safe_state = SAFE_CHECKPOINT["model_state_dict"]
        mapped_safe_state = {
            ("lstm." + key[len("context."):]) if key.startswith("context.") else key: value
            for key, value in safe_state.items()
        }
        SAFE_CONTEXT = ContextBiLSTM(n_labels=NUM_LABELS).to(DEVICE)
        safe_load_result = SAFE_CONTEXT.load_state_dict(mapped_safe_state, strict=True)
        assert not safe_load_result.missing_keys and not safe_load_result.unexpected_keys
        for domain in ("CL", "IT"):
            SAFE_DEV_LOGITS.update(predict_document_logits(SAFE_CONTEXT, V1_CONTEXT_FEATURES[domain]["dev"]))
            SAFE_TEST_LOGITS.update(predict_document_logits(SAFE_CONTEXT, V1_CONTEXT_FEATURES[domain]["test"]))
        SAFE_DEV_METRICS = {
            domain: metrics_for_domain(SAFE_DEV_LOGITS, SPLITS[domain]["dev"])
            for domain in ("CL", "IT")
        }
        SAFE_DEV_MEAN = mean_domain_score(SAFE_DEV_METRICS)
        expected_safe_mean = float(SAFE_CHECKPOINT["candidate_dev_score"])
        SAFE_REFERENCE_VERIFIED = abs(SAFE_DEV_MEAN - expected_safe_mean) <= 0.005
        SAFE_REFERENCE_AVAILABLE = True
        print("V1 safe-retrain checkpoint recorded development mean:", round(expected_safe_mean, 6))
        print("V1 safe-retrain replayed development mean:", round(SAFE_DEV_MEAN, 6))
        print("V1 safe-retrain reference verified:", SAFE_REFERENCE_VERIFIED)

        if FILES["phase3_v1_safe_retrain_prediction.json"] is not None:
            saved_safe = read_json(FILES["phase3_v1_safe_retrain_prediction.json"])
            if prediction_json_is_aligned(saved_safe, SPLITS):
                SAVED_SAFE_PREDICTION = saved_safe
                safe_reproduced = {
                    doc_id: [ID_TO_LABEL[int(i)] for i in logits.argmax(-1)]
                    for doc_id, logits in SAFE_TEST_LOGITS.items()
                }
                n_safe_mismatch = sum(
                    safe_reproduced[doc_id] != saved_safe[doc_id]
                    for doc_id in safe_reproduced
                )
                SAFE_TEST_MATCHES_SAVED = n_safe_mismatch == 0
                print("V1 safe-retrain recomputation vs saved prediction documents with label differences:",
                      n_safe_mismatch)
            else:
                print("Safe-retrain prediction file is attached but fails the test contract; blend disabled.")
else:
    print("V1 reference comparison skipped: attach both v1_model.pt and best_phase3_v1_context_bilstm.pt.")


## 10. Development-only selection, final predictions, and Codabench ZIP

The guard uses the strongest prior development reference: the accepted V1 safe-retrain checkpoint when its score replays, otherwise audited safe-retrain development metrics recorded with this project. A new V4 model needs at least +0.01 mean fixed-13 Macro-F1 and cannot fall by more than 0.01 in either domain. A fixed 50:50 blend needs at least +0.005 mean and must also stay within 0.01 per domain. Model and blend selection use development data only. Test prediction comparisons are logged as provenance checks and never influence the chosen model. The notebook always writes a V4-only candidate ZIP for an optional research submission; `submission.zip` follows the development guard and falls back to the best aligned saved incumbent when available.

In [ ]:
def logits_to_prediction(logits_by_id):
    return {doc_id: [ID_TO_LABEL[int(x)] for x in logits.argmax(-1)]
            for doc_id, logits in logits_by_id.items()}

def probabilities(logits_by_id):
    return {doc_id: torch.softmax(logits, dim=-1) for doc_id, logits in logits_by_id.items()}

def metric_for_probability_map(probability_by_id, domain):
    as_logits = {doc_id: probability.log() for doc_id, probability in probability_by_id.items()}
    return metrics_for_domain(as_logits, SPLITS[domain]["dev"])

V4_DEV_PREDICTION = logits_to_prediction(V4_DEV_LOGITS)
V4_TEST_PREDICTION = logits_to_prediction(V4_TEST_LOGITS)
V4_DEV_METRICS = {d: metrics_for_domain(V4_DEV_LOGITS, SPLITS[d]["dev"]) for d in ("CL", "IT")}
V4_DEV_MEAN = mean_domain_score(V4_DEV_METRICS)

SELECTION_GUARD = {
    "minimum_mean_development_gain_over_reference": 0.01,
    "maximum_allowed_development_drop_per_domain": 0.01,
    "minimum_blend_mean_development_gain_over_reference": 0.005,
    "ensemble_rule": "fixed equal-weight mean of verified reference and V4 softmax probabilities; no test-based tuning",
}
if SAFE_REFERENCE_VERIFIED:
    REFERENCE_NAME = "v1_safe_retrain"
    REFERENCE_DEV_LOGITS, REFERENCE_TEST_LOGITS = SAFE_DEV_LOGITS, SAFE_TEST_LOGITS
    REFERENCE_DEV_METRICS = SAFE_DEV_METRICS
    REFERENCE_TEST_MATCHES_SAVED = SAFE_TEST_MATCHES_SAVED
else:
    # This accepted prior run was audited from its checkpoint and metrics file.
    # Keep the comparison guard against it even if its large checkpoint is not attached in Kaggle.
    REFERENCE_NAME = "v1_safe_retrain_recorded_development_metrics"
    REFERENCE_DEV_LOGITS, REFERENCE_TEST_LOGITS = {}, {}
    REFERENCE_DEV_METRICS = {
        "CL": {"macro_f1_fixed_13": 0.3870055167420532},
        "IT": {"macro_f1_fixed_13": 0.4343563782142648},
    }
    REFERENCE_TEST_MATCHES_SAVED = False
chosen_source = "v4_context_bilstm"
chosen_prediction = V4_TEST_PREDICTION
selection_record = {
    "selection_metric": "unweighted mean of fixed-13-label CL and IT development Macro-F1",
    "v4_dev_by_domain": {d: V4_DEV_METRICS[d]["macro_f1_fixed_13"] for d in ("CL", "IT")},
    "v4_dev_mean": V4_DEV_MEAN,
    "v1_reference_available": V1_REFERENCE_AVAILABLE,
    "v1_reference_reproduced_dev": V1_REFERENCE_VERIFIED,
    "v1_test_predictions_match_saved_incumbent": V1_TEST_MATCHES_SAVED,
    "safe_retrain_reference_available": SAFE_REFERENCE_AVAILABLE,
    "safe_retrain_reference_reproduced_dev": SAFE_REFERENCE_VERIFIED,
    "safe_retrain_test_predictions_match_saved_file": SAFE_TEST_MATCHES_SAVED,
    "comparison_reference": REFERENCE_NAME,
    "guard": SELECTION_GUARD,
}

if REFERENCE_NAME is not None:
    base_domain = {d: REFERENCE_DEV_METRICS[d]["macro_f1_fixed_13"] for d in ("CL", "IT")}
    base_mean = mean_domain_score(REFERENCE_DEV_METRICS)
    candidate_passes = (
        V4_DEV_MEAN >= base_mean + SELECTION_GUARD["minimum_mean_development_gain_over_reference"]
        and all(V4_DEV_METRICS[d]["macro_f1_fixed_13"] >= base_domain[d] -
                SELECTION_GUARD["maximum_allowed_development_drop_per_domain"] for d in ("CL", "IT"))
    )
    selection_record["reference_dev_by_domain"] = base_domain
    selection_record["reference_dev_mean"] = base_mean
    selection_record["v4_candidate_passes_guard"] = candidate_passes

    # A blend is allowed only with a live, development-verified reference; its
    # promotion is determined by development metrics, never by test predictions.
    can_blend = bool(REFERENCE_TEST_LOGITS) and (SAFE_REFERENCE_VERIFIED or V1_REFERENCE_VERIFIED)
    if can_blend:
        reference_probs, v4_probs = probabilities(REFERENCE_DEV_LOGITS), probabilities(V4_DEV_LOGITS)
        blend_dev_probs = {doc_id: 0.5 * reference_probs[doc_id] + 0.5 * v4_probs[doc_id]
                           for doc_id in v4_probs}
        blend_dev_metrics = {d: metric_for_probability_map(blend_dev_probs, d) for d in ("CL", "IT")}
        blend_mean = mean_domain_score(blend_dev_metrics)
        blend_passes = (
            blend_mean >= base_mean + SELECTION_GUARD["minimum_blend_mean_development_gain_over_reference"]
            and all(blend_dev_metrics[d]["macro_f1_fixed_13"] >= base_domain[d] - 0.01 for d in ("CL", "IT"))
        )
        selection_record["equal_blend_dev_by_domain"] = {
            d: blend_dev_metrics[d]["macro_f1_fixed_13"] for d in ("CL", "IT")
        }
        selection_record["equal_blend_dev_mean"] = blend_mean
        selection_record["equal_blend_passes_guard"] = blend_passes
        if candidate_passes and (not blend_passes or V4_DEV_MEAN >= blend_mean):
            chosen_source, chosen_prediction = "v4_context_bilstm", V4_TEST_PREDICTION
        elif blend_passes:
            reference_test_probs, v4_test_probs = probabilities(REFERENCE_TEST_LOGITS), probabilities(V4_TEST_LOGITS)
            blend_test_probs = {doc_id: 0.5 * reference_test_probs[doc_id] +
                                         0.5 * v4_test_probs[doc_id]
                                for doc_id in V4_TEST_LOGITS}
            chosen_prediction = {
                doc_id: [ID_TO_LABEL[int(x)] for x in probability.argmax(-1)]
                for doc_id, probability in blend_test_probs.items()
            }
            chosen_source = "fixed_50_50_v1_v4_probability_blend"
        elif candidate_passes:
            chosen_source, chosen_prediction = "v4_context_bilstm", V4_TEST_PREDICTION
        else:
            if SAVED_INCUMBENT is not None:
                chosen_source, chosen_prediction = "saved_v1_04311_incumbent", SAVED_INCUMBENT
            elif SAFE_REFERENCE_VERIFIED:
                chosen_source = "recomputed_v1_safe_retrain_reference"
                chosen_prediction = logits_to_prediction(SAFE_TEST_LOGITS)
            elif V1_REFERENCE_VERIFIED:
                chosen_source = "recomputed_v1_original_context_reference"
                chosen_prediction = logits_to_prediction(V1_TEST_LOGITS)
            else:
                chosen_source, chosen_prediction = "v4_context_bilstm_unverified_v1_fallback", V4_TEST_PREDICTION
    elif candidate_passes:
        chosen_source, chosen_prediction = "v4_context_bilstm", V4_TEST_PREDICTION
    else:
        if SAVED_INCUMBENT is not None:
            chosen_source, chosen_prediction = "saved_v1_04311_incumbent", SAVED_INCUMBENT
        elif SAFE_REFERENCE_VERIFIED:
            chosen_source = "recomputed_v1_safe_retrain_reference"
            chosen_prediction = logits_to_prediction(SAFE_TEST_LOGITS)
        elif V1_REFERENCE_VERIFIED:
            chosen_source = "recomputed_v1_original_context_reference"
            chosen_prediction = logits_to_prediction(V1_TEST_LOGITS)
        else:
            chosen_source, chosen_prediction = "v4_context_bilstm_unverified_v1_fallback", V4_TEST_PREDICTION
selection_record["selected_test_prediction_source"] = chosen_source
selection_record["test_labels_used_for_selection"] = False
selection_record["public_score"] = None  # Must be filled only after Codabench returns a result.

expected_test_ids = {doc["id"] for domain in ("CL", "IT") for doc in SPLITS[domain]["test"]}
if set(chosen_prediction) != expected_test_ids:
    raise ValueError("Selected prediction IDs do not exactly match the two supplied test splits")
for domain in ("CL", "IT"):
    for doc in SPLITS[domain]["test"]:
        labels = chosen_prediction[doc["id"]]
        if len(labels) != len(doc["text"]):
            raise ValueError(f"{doc['id']}: prediction count differs from input sentence count")
        if any(label not in LABEL_TO_ID for label in labels):
            raise ValueError(f"{doc['id']}: prediction contains a label outside label_vocab.json")

def write_json(path, obj):
    with open(path, "w", encoding="utf-8") as stream:
        json.dump(obj, stream, ensure_ascii=False, indent=2)

final_prediction_path = OUTPUT_DIR / "prediction.json"
submission_zip_path = OUTPUT_DIR / "submission.zip"
write_json(final_prediction_path, chosen_prediction)
write_json(OUTPUT_DIR / "phase3_v4_candidate_prediction.json", V4_TEST_PREDICTION)
write_json(OUTPUT_DIR / "phase3_v4_model_comparison.json", selection_record)
write_json(OUTPUT_DIR / "phase3_v4_dev_metrics.json", {
    "v4_candidate": V4_DEV_METRICS,
    "v4_candidate_mean_domain_macro_f1_fixed_13": V4_DEV_MEAN,
    "v1_reference": V1_DEV_METRICS,
    "v1_reference_mean_domain_macro_f1_fixed_13": mean_domain_score(V1_DEV_METRICS) if V1_DEV_METRICS else None,
    "v1_safe_retrain_reference_dev_mean_fixed_13": mean_domain_score(SAFE_DEV_METRICS) if SAFE_DEV_METRICS else 0.410680947478159,
    "comparison_reference_dev_by_domain": selection_record.get("reference_dev_by_domain"),
    "selection": selection_record,
})
with zipfile.ZipFile(submission_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(final_prediction_path, arcname="prediction.json")
v4_candidate_zip_path = OUTPUT_DIR / "phase3_v4_candidate_submission.zip"
with zipfile.ZipFile(v4_candidate_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(OUTPUT_DIR / "phase3_v4_candidate_prediction.json", arcname="prediction.json")
shutil.copy2(submission_zip_path, Path("/kaggle/working/submission.zip"))

with zipfile.ZipFile(submission_zip_path, "r") as archive:
    assert archive.namelist() == ["prediction.json"], archive.namelist()
    assert archive.testzip() is None, "ZIP integrity test failed"
    zipped_prediction = json.loads(archive.read("prediction.json").decode("utf-8"))
    assert zipped_prediction == chosen_prediction
with zipfile.ZipFile(v4_candidate_zip_path, "r") as archive:
    assert archive.namelist() == ["prediction.json"], archive.namelist()
    assert archive.testzip() is None, "V4 candidate ZIP integrity test failed"
    assert json.loads(archive.read("prediction.json").decode("utf-8")) == V4_TEST_PREDICTION
assert prediction_json_is_aligned(chosen_prediction, SPLITS)
print("\nSelected prediction source:", chosen_source)
print("Submission file:", final_prediction_path)
print("Codabench ZIP:", submission_zip_path)
print("V4-only candidate ZIP (for explicit research evaluation):", v4_candidate_zip_path)
print("ZIP contains exactly one root file: prediction.json")
print("Test ID, per-document sentence count, and label checks: passed")
print("No public score is reported by this notebook; score must come from Codabench.")


## 11. What to inspect after the Kaggle run

Download `submission.zip` from Kaggle's Output tab and submit that ZIP to Codabench. The ZIP is checked in the notebook to contain exactly one root-level `prediction.json`, with the supplied test IDs, sentence counts, and legal labels.

Also download these files before comparing runs:

- `phase3_v4_model_comparison.json` — selected prediction source and development-only guard values.
- `phase3_v4_dev_metrics.json` — fixed-13 and gold-supported diagnostic metrics, confusion matrices, and per-label support.
- `phase3_v4_training_history.json` — context-head epoch history.
- `phase3_v4_candidate_prediction.json` and `phase3_v4_candidate_submission.zip` — V4-only predictions, kept separately for an optional research submission.
- `best_phase3_v4_context_bilstm.pt` — selected document-context head (the V4 encoder remains the supplied Phase 1 checkpoint).

The Codabench score is a separate hidden-test result. Do not call the development score the public score, and do not update the model-selection rule after seeing the test labels (they are not provided). `submission.zip` follows the comparison guard. Use `phase3_v4_candidate_submission.zip` only if you explicitly want to score the V4-only candidate despite a failed guard.